# Zaskaleta AI Twin — AUTO v4

## ▶ ЗАПУСК З ТЕЛЕФОНУ
**Runtime → Run all.**

AUTO v4 тепер працює з двома ПОСТІЙНИМИ Google Drive папками за ID, а не з `MyDrive` поточного акаунта.

**SOURCE / робоча папка:** `13Wye5lVZPOcUryXbXhplad_FkxM7lG4a`

**ARCHIVE / готові результати:** `1_7G-rAGQ80Vpe_CWdGOzPIg0nuprDp3s`

Можна запускати Colab під іншим Google-акаунтом через GPU-ліміти. Поточний акаунт повинен мати доступ до обох цих папок.

**T4 GPU має бути увімкнений.**


In [ ]:
import os, re, subprocess, sys
from pathlib import Path
import torch

SOURCE_FOLDER_ID='13Wye5lVZPOcUryXbXhplad_FkxM7lG4a'
ARCHIVE_FOLDER_ID='1_7G-rAGQ80Vpe_CWdGOzPIg0nuprDp3s'
SOURCE_LOCAL=Path('/content/zaskaleta_fixed_source')
ROOT=Path('/content/zaskaleta-ai-twin-colab')

GPU_OK=torch.cuda.is_available()
print('CUDA:',GPU_OK)
if not GPU_OK:
    print('\n⛔ GPU ЗАРАЗ НЕДОСТУПНИЙ У COLAB')
    print('Нічого не запускаємо на CPU.')
    raise SystemExit(0)
print('✅ T4/CUDA доступний — продовжуємо AUTO v4')

# Clone code first; private media never goes to GitHub.
if ROOT.exists():
    subprocess.run(['rm','-rf',str(ROOT)],check=True)
subprocess.run(['git','clone','--depth','1','https://github.com/sergokharkov/zaskaleta-ai-twin-colab.git',str(ROOT)],check=True)
WORKER=ROOT/'worker'

# Google Drive API auth for whichever Google account is currently used in Colab.
from google.colab import auth
print('\n🔐 GOOGLE DRIVE API')
print('Авторизуй ПОТОЧНИЙ Google-акаунт. AUTO звертатиметься прямо до двох фіксованих папок за ID.')
auth.authenticate_user()
try:
    import googleapiclient.discovery
except Exception:
    subprocess.run([sys.executable,'-m','pip','install','-q','google-api-python-client','google-auth','google-auth-httplib2'],check=True)

# Pull canonical source folder to the ephemeral Colab runtime.
if SOURCE_LOCAL.exists():
    subprocess.run(['rm','-rf',str(SOURCE_LOCAL)],check=True)
SOURCE_LOCAL.mkdir(parents=True,exist_ok=True)
print('\n📥 Синхронізація ПОСТІЙНОЇ SOURCE-папки...')
pull=subprocess.run([sys.executable,str(WORKER/'fixed_drive_folder_sync.py'),'pull','--folder-id',SOURCE_FOLDER_ID,'--local-dir',str(SOURCE_LOCAL)])
if pull.returncode != 0:
    print('⛔ Поточний Google-акаунт не має доступу до SOURCE-папки.')
    print('Надай цьому акаунту доступ до папки 13Wye5lVZPOcUryXbXhplad_FkxM7lG4a і запусти знову.')
    raise SystemExit(0)

# Check archive access now, before expensive GPU work. Rendering may continue if only archive access is missing.
archive_access=subprocess.run([sys.executable,str(WORKER/'fixed_drive_folder_sync.py'),'check','--folder-id',ARCHIVE_FOLDER_ID,'--local-dir','/content/zaskaleta_archive_check']).returncode == 0
if archive_access:
    print('✅ Постійна ARCHIVE-папка доступна')
else:
    print('⚠️ ARCHIVE-папка зараз недоступна цьому акаунту. Рендер продовжиться, але фінал не зможе автоматично заархівуватися.')

# Fast source preflight before installer.
voice_matches=[]
for ext in ('mp3','wav','m4a','flac','aac','ogg'):
    voice_matches += list(SOURCE_LOCAL.rglob(f'Zaskaleta_AI_Voice_Master.{ext}'))
if not voice_matches:
    print('⛔ У фіксованій SOURCE-папці не знайдено Zaskaleta_AI_Voice_Master.*')
    raise SystemExit(0)
print('✅ Fixed SOURCE preflight OK:',voice_matches[0])

env=os.environ.copy()
env['APP_DIR']=str(WORKER)
env['MUSETALK_ROOT']='/content/MuseTalk'
env['VENV_DIR']='/content/ai-twin-py311'
env['PRIMARY_DRIVE']=str(SOURCE_LOCAL)

print('\n========== INSTALLER START ==========')
iproc=subprocess.Popen(['bash',str(WORKER/'install_gpu_engines.sh')],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
install_lines=[]
for line in iproc.stdout:
    print(line,end='')
    install_lines.append(line)
icode=iproc.wait()
if icode != 0:
    tail=''.join(install_lines[-80:])
    print('\n❌ INSTALLER FAILED — last output:\n'+tail)
    raise RuntimeError(f'Installer stopped with exit code {icode}')
print('========== INSTALLER OK ==========\n')

PY='/content/ai-twin-py311/bin/python'
cmd=[PY,str(WORKER/'run_auto_v3.py'),'--root',str(ROOT),'--mydrive',str(SOURCE_LOCAL),'--sync-folder-id',SOURCE_FOLDER_ID]
proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
lines=[]
for line in proc.stdout:
    print(line,end='')
    lines.append(line)
code=proc.wait()
if code != 0:
    print('\n❌ AUTO PIPELINE FAILED — last output:\n'+''.join(lines[-80:]))
    raise RuntimeError(f'AUTO v4 stopped with exit code {code}')
out=''.join(lines)
m=re.search(r'^FINAL_PATH=(.+)$',out,re.MULTILINE)
if not m:
    raise RuntimeError('AUTO v4 завершився без FINAL_PATH')
FINAL=m.group(1).strip()
final_path=Path(FINAL)
print('✅ FINAL READY:',FINAL)

# Always target the one fixed archive folder. No folder prompt and no account switching.
if archive_access:
    try:
        day_match=re.search(r'Day_(\d{2})',final_path.name) or re.search(r'Day_(\d{2})',str(final_path.parent))
        archive_day=int(day_match.group(1)) if day_match else 0
        subprocess.run([sys.executable,str(WORKER/'archive_second_drive.py'),'--folder',ARCHIVE_FOLDER_ID,'--episode-dir',str(final_path.parent),'--final',str(final_path),'--day',str(archive_day)],check=True)
        print('✅ Готове відео та метадані збережено в ПОСТІЙНІЙ ARCHIVE-папці')
    except Exception as e:
        print('⚠️ Архівація не вдалася, але прогрес уже синхронізовано назад у SOURCE-папку.')
        print('Причина:',e)

from IPython.display import Video, display
print('✅ FINAL:',FINAL)
display(Video(FINAL,embed=True,width=360))
